# VulBERTa LoRA Notebook

A standalone notebook for loading a VulBERTa checkpoint and applying PEFT LoRA adapters.

Default checkpoint: `claudios/VulBERTa-MLP-D2A`.
Set `VULBERTA_MODEL` to use a local path or a different Hugging Face repo.

In [1]:
# Remove incompatible torchao before loading PEFT.
# This keeps the full workflow inside the notebook.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torchao') is not None:
    print('Removing incompatible torchao package...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

# Uncomment if the notebook kernel does not already have these packages.
# %pip install -q --upgrade transformers accelerate peft datasets sentencepiece

import os
import torch
from dataclasses import dataclass

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

Removing incompatible torchao package...
torch: 2.10.0+cu128
cuda available: True


## Load VulBERTa

This notebook defaults to a VulBERTa checkpoint and keeps the LoRA targets

In [2]:
DEFAULT_MODEL_NAME = os.environ.get('VULBERTA_MODEL', 'claudios/VulBERTa-MLP-D2A')
DEFAULT_MAX_LENGTH = 128
DEFAULT_NUM_LABELS = 2

@dataclass(frozen=True)
class LoRASettings:
    r: int = 8
    alpha: int = 32
    dropout: float = 0.1
    num_labels: int = DEFAULT_NUM_LABELS
    max_length: int = DEFAULT_MAX_LENGTH


def suggest_target_modules(model):
    module_names = {name.split('.')[-1] for name, _ in model.named_modules()}
    preferred_groups = [
        ['query', 'key', 'value'],
        ['query', 'value'],
        ['q_proj', 'k_proj', 'v_proj'],
        ['q_proj', 'v_proj'],
    ]
    for group in preferred_groups:
        if all(module_name in module_names for module_name in group):
            return group
    fallback = [name for name in ['query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj'] if name in module_names]
    return fallback or ['query', 'value']


def load_vulberta(model_name=DEFAULT_MODEL_NAME, num_labels=DEFAULT_NUM_LABELS):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    model.config.label2id = {str(index): index for index in range(num_labels)}
    model.config.id2label = {index: str(index) for index in range(num_labels)}
    return tokenizer, model

## Apply LoRA

The adapter targets the attention projection names exposed by the VulBERTa backbone.

In [3]:
def build_lora_model(base_model, settings=LoRASettings(), target_modules=None):
    modules = list(target_modules) if target_modules is not None else suggest_target_modules(base_model)
    config = LoraConfig(
        r=settings.r,
        lora_alpha=settings.alpha,
        lora_dropout=settings.dropout,
        target_modules=modules,
        bias='none',
        task_type=TaskType.SEQ_CLS,
    )
    return get_peft_model(base_model, config)


def save_adapter(model, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)


def load_adapter_for_inference(model_name, adapter_dir, num_labels=DEFAULT_NUM_LABELS):
    tokenizer, base_model = load_vulberta(model_name=model_name, num_labels=num_labels)
    adapted_model = PeftModel.from_pretrained(base_model, adapter_dir)
    adapted_model.eval()
    return tokenizer, adapted_model

## Smoke Test

Load the base checkpoint, wrap it with LoRA, and run a tiny prediction example.

In [4]:
def predict(tokenizer, model, texts, max_length=DEFAULT_MAX_LENGTH):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        return_tensors='pt',
        max_length=max_length,
    )
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)
    return predictions.cpu(), probabilities.cpu()


def smoke_test(model_name=DEFAULT_MODEL_NAME):
    tokenizer, base_model = load_vulberta(model_name=model_name)
    lora_model = build_lora_model(base_model)

    trainable_params = sum(parameter.numel() for parameter in lora_model.parameters() if parameter.requires_grad)
    total_params = sum(parameter.numel() for parameter in lora_model.parameters())
    print(f'Loaded: {model_name}')
    print(f'Target modules: {suggest_target_modules(base_model)}')
    print(f'Trainable params: {trainable_params} / {total_params} ({100 * trainable_params / total_params:.2f}%)')

    sample_texts = [
        'Potential buffer overflow in copy routine.',
        'Helper function for formatting dates.',
    ]
    predictions, probabilities = predict(tokenizer, lora_model, sample_texts)
    print('Predictions:', predictions.tolist())
    print('Probabilities:', probabilities.tolist())


smoke_test()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded: claudios/VulBERTa-MLP-D2A
Target modules: ['query', 'key', 'value']
Trainable params: 1034498 / 125871364 (0.82%)
Predictions: [1, 0]
Probabilities: [[0.20570628345012665, 0.7942937016487122], [0.7996674180030823, 0.20033258199691772]]


In [6]:
from transformers import AutoTokenizer, AutoModel

GCB_MODEL_NAME = "microsoft/graphcodebert-base"
gcb_tokenizer = AutoTokenizer.from_pretrained(GCB_MODEL_NAME)
gcb_encoder = AutoModel.from_pretrained(GCB_MODEL_NAME)

# Reload VulBERTa tokenizer since it's not in scope
vulberta_tokenizer, _ = load_vulberta()

print("✓ GraphCodeBERT tokenizer and encoder loaded")
print(f"VulBERTa vocab size:      {len(vulberta_tokenizer)}")
print(f"GraphCodeBERT vocab size: {len(gcb_tokenizer)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ GraphCodeBERT tokenizer and encoder loaded
VulBERTa vocab size:      50000
GraphCodeBERT vocab size: 50265


#Build dual input model


In [9]:
import torch.nn as nn

class VulBERTaDualInputModel(nn.Module):
    def __init__(self, vulberta_model, gcb_encoder):
        super().__init__()
        self.vulberta = vulberta_model
        self.gcb = gcb_encoder

    def forward(
        self,
        vulberta_input_ids,
        vulberta_attention_mask,
        gcb_input_ids,
        gcb_attention_mask,
    ):
        # VulBERTa forward pass → go deeper to get hidden states
        vulberta_out = self.vulberta.base_model.roberta(
            input_ids=vulberta_input_ids,
            attention_mask=vulberta_attention_mask,
        )
        vulberta_cls = vulberta_out.last_hidden_state[:, 0, :]  # [B, 768]

        # GraphCodeBERT forward pass → get CLS token
        gcb_out = self.gcb(
            input_ids=gcb_input_ids,
            attention_mask=gcb_attention_mask,
        )
        gcb_cls = gcb_out.last_hidden_state[:, 0, :]  # [B, 768]

        return {
            "vulberta_embedding": vulberta_cls,
            "gcb_embedding": gcb_cls,
        }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build the dual model using already loaded models
_, base_model = load_vulberta()
lora_model = build_lora_model(base_model)
gcb_encoder = gcb_encoder.to(device)

dual_model = VulBERTaDualInputModel(
    vulberta_model=lora_model.to(device),
    gcb_encoder=gcb_encoder,
).to(device)

print(f"✓ Dual input model ready on {device}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Dual input model ready on cuda


testing:


In [11]:
sample_code = "def get_user(uid): return db.execute('SELECT * FROM users WHERE id=' + uid)"

MAX_LENGTH = 128

# Tokenize with VulBERTa
vulberta_inputs = vulberta_tokenizer(
    sample_code,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

# Tokenize with GraphCodeBERT
gcb_inputs = gcb_tokenizer(
    sample_code,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

# Move to device
vulberta_inputs = {k: v.to(device) for k, v in vulberta_inputs.items()}
gcb_inputs = {k: v.to(device) for k, v in gcb_inputs.items()}

# Run forward pass
dual_model.eval()
with torch.no_grad():
    output = dual_model(
        vulberta_input_ids=vulberta_inputs["input_ids"],
        vulberta_attention_mask=vulberta_inputs["attention_mask"],
        gcb_input_ids=gcb_inputs["input_ids"],
        gcb_attention_mask=gcb_inputs["attention_mask"],
    )

print("✓ Dual input forward pass successful!")
print(f"VulBERTa embedding shape: {output['vulberta_embedding'].shape}")
print(f"GCB embedding shape:      {output['gcb_embedding'].shape}")
print("\n✅ Ready for fusion layer!")

✓ Dual input forward pass successful!
VulBERTa embedding shape: torch.Size([1, 768])
GCB embedding shape:      torch.Size([1, 768])

✅ Ready for fusion layer!
